# Smart Semsar — furniture demo on Colab (Gradio)

تجربة كل خطوات الفرش على إعلان واحد: المخطط → العفش → المراجعة → جولة 3D، وكمان قطعة فرش من صورتك.

**What it does:** opens the Gradio demo (`app.py`) with a public link: pick a listing, set your wishes,
upload a photo of your own furniture, and walk through the furnished 3D plan.

| In the app | Needs |
|---|---|
| listing furnished by the rules, with your wishes | nothing |
| **your own piece from a photo**, made 3D by TripoSR **inside this notebook** and placed in the room | nothing (no account, no quota) |
| your own piece from a `.glb` file | nothing |
| furniture chosen and arranged by AI ("Who arranges the furniture") | `GEMINI_API_KEY` and/or `GROQ_API_KEY` |
| the photo read automatically (what it is, its width) | `GEMINI_API_KEY` or `GROQ_API_KEY` |

**Photo → 3D** runs TripoSR right here (step 5). The model files are public, so no Hugging Face token or
GPU quota is involved. Works on the normal CPU runtime (about 1 min per photo); with
`Runtime → Change runtime type → T4 GPU` it takes a few seconds.
TRELLIS.2 on Hugging Face is still offered in the app, but its free daily GPU quota runs out quickly.

**Before you run** — optional keys: open the key icon (Secrets) on the left, add with *Notebook access* on:
- `GEMINI_API_KEY` — free at https://aistudio.google.com/apikey — reads the photo, AI planner
- `GROQ_API_KEY` — free at https://console.groq.com/keys — takes over when Gemini is busy
- `HF_TOKEN` — only for the TRELLIS.2 option
- `GITHUB_TOKEN` — only if the repository is private (a token with read access to it)

Then `Runtime → Run all`.

## 1. Settings

In [ ]:
REPO = "ghadeersalah42/SmartSemsar"
BRANCH = "furniture-merge"      # the branch with the merged furniture work
PROPERTY_ID = "PROP_1002"       # any property_id from data/final_merged_dataset.csv

## 2. Get the code
Clones the branch from GitHub. If the branch is not on GitHub yet, upload `furniture-merge.bundle`
when asked (it is applied on top of `3D-Plan`).

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:          # secret missing or notebook access off
        return ""

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
os.chdir("/content")
if os.path.isdir("SmartSemsar"):        # an older copy in this runtime: bring it up to date
    got = subprocess.run(["git", "-C", "SmartSemsar", "fetch", "-q", "origin", BRANCH]).returncode == 0
    if got:
        subprocess.run(["git", "-C", "SmartSemsar", "reset", "-q", "--hard", "FETCH_HEAD"], check=True)
    else:
        print("Could not update the existing copy from GitHub; if you changed the bundle, "
              "use Runtime > Disconnect and delete runtime, then Run all.")
else:
    ok = subprocess.run(["git", "clone", "-q", "--depth", "50", "-b", BRANCH, url, "SmartSemsar"]).returncode == 0
    if not ok:
        print(f"Branch {BRANCH} not found on GitHub: cloning 3D-Plan and applying the bundle instead.")
        subprocess.run(["git", "clone", "-q", "-b", "3D-Plan", url, "SmartSemsar"], check=True)
        from google.colab import files
        print("Upload furniture-merge.bundle:")
        name = next(iter(files.upload()))
        subprocess.run(["git", "-C", "SmartSemsar", "fetch", "-q", f"/content/{name}", f"{BRANCH}:{BRANCH}"], check=True)
        subprocess.run(["git", "-C", "SmartSemsar", "checkout", "-q", BRANCH], check=True)
os.chdir("/content/SmartSemsar")
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

## 3. Install (1-2 minutes)

In [ ]:
!pip install -q shapely trimesh scipy mapbox_earcut pandas pydantic pillow requests gradio gradio_client
!pip install -q -r requirements-photo3d.txt     # TripoSR's packages; Colab already has torch and transformers

## 4. Keys
Read from Colab Secrets into the environment; only whether each one is set is printed.

In [ ]:
for name in ("HF_TOKEN", "GEMINI_API_KEY", "GROQ_API_KEY"):
    pasted = secret(name)
    value = "".join(pasted.split())          # keys never contain spaces or line breaks
    if value:
        os.environ[name] = value
    note = " (had spaces or line breaks from pasting; cleaned)" if value != pasted.strip() else ""
    print(f"{name:15s} {'set' if value else 'not set'}{note}")

## 5. Photo → 3D: load TripoSR and test it
Downloads TripoSR (1.7 GB, public files, no token) and turns its example chair photo into a 3D model.
First time: 1–3 minutes. If this cell prints an error, the app still works with `.glb` files.

In [ ]:
import sys, time
sys.path.insert(0, "/content/SmartSemsar")
from huggingface_hub import hf_hub_download
from backend.services import triposr_local

print("device:", triposr_local.device(), "| missing packages:", triposr_local.missing_packages() or "none")
t0 = time.time()
try:
    triposr_local.load_model()
    print(f"model loaded in {time.time() - t0:.0f} s")
    example = hf_hub_download("stabilityai/TripoSR", "examples/chair_p.webp", repo_type="space", token=False)
    t0 = time.time()
    glb = triposr_local.generate_model(example, "/content/test_chair.glb", say=print)
    print(f"test chair: {glb.stat().st_size // 1024} KB in {time.time() - t0:.0f} s  -> photo to 3D works")
except triposr_local.TripoSRError as e:
    print("error:", e)

## 6. Open the Gradio app
Starts the demo and prints a public `https://….gradio.live` link (valid while this notebook runs).
The app also shows below. Photo → 3D takes about 1 min on CPU, a few seconds on a GPU.

In [ ]:
import os, sys
import gradio as gr
os.chdir("/content/SmartSemsar")
sys.path.insert(0, "/content/SmartSemsar")
for name in [m for m in sys.modules if m.split(".")[0] in ("app", "inference", "backend")
             and m != "backend.services.triposr_local"]:      # keeps the TripoSR model loaded in step 5
    del sys.modules[name]      # re-running this cell always uses the current code and keys
gr.close_all()
import app
print(os.popen("git log --oneline -1").read())
app.demo.queue().launch(share=True, height=1100)

---
## Optional: the same runs from the command line
`inference.py` prints each step; the walkthrough is shown inside the notebook.

### Viewer
Serves the `out/` folder on a local port and shows a walkthrough page inside the notebook.
In the viewer: drag to look around, **Walk** + W A S D to move, or open it full screen with the link.

In [ ]:
import subprocess, time
from google.colab import output

server = subprocess.Popen(["python", "-m", "http.server", "8001", "--directory", "out"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.makedirs("out", exist_ok=True)
time.sleep(1)

def show(property_id=PROPERTY_ID, height=720):
    page = f"/{property_id}/walkthrough.html"
    if not os.path.exists("out" + page):
        print("No walkthrough built for", property_id, "- see the messages above.")
        return
    output.serve_kernel_port_as_iframe(8001, path=page, height=height)

### Run A — the listing furnished by the rules

In [ ]:
!python inference.py $PROPERTY_ID --no-open
show()

### Run B — with your wishes
`--density minimal|normal|full`, `--must-have` / `--exclude` catalog names (sofa, desk, tv, armchair ...),
`--dining-seats 4|6`, `--style-brief` free text.

In [ ]:
!python inference.py $PROPERTY_ID --density full --must-have desk armchair --exclude tv --dining-seats 4 --no-open
show()

### Run C — your own furniture from a photo (TripoSR, in this notebook)
Upload a photo of **one** piece, fully visible (a plain background helps).
Set `KIND` (sofa, armchair, bed, desk, ...) and its real `WIDTH_M`; with `GEMINI_API_KEY` set you can leave
`KIND = ""` and the photo is recognised. About 1 min on CPU. `--generator trellis` would use TRELLIS.2 instead.

In [ ]:
from google.colab import files

KIND = "armchair"     # "" = let Gemini read it from the photo
WIDTH_M = 0.8         # real width in meters; 0 = the catalog piece's width

uploaded = files.upload()
PHOTO = "/content/" + next(iter(uploaded))
with open(PHOTO, "wb") as f:
    f.write(uploaded[next(iter(uploaded))])
opts = (f"--kind {KIND} " if KIND else "") + (f"--width {WIDTH_M} " if WIDTH_M else "")
!python inference.py $PROPERTY_ID --photo "$PHOTO" $opts --no-open
show()

### Run D — Gemini chooses and arranges the furniture
The model writes placement instructions ("sofa on wall B"); the code computes the positions and the
validator checks them. Without `GEMINI_API_KEY` this falls back to the rules and says so.

In [ ]:
STYLE_BRIEF = "warm and bright, light wood, a reading corner in the living room"
!python inference.py $PROPERTY_ID --planner llm --style-brief "$STYLE_BRIEF" --no-open
show()

## Download the results
`staging.json`, `walkthrough.html` (opens offline in any browser) and your custom model.

In [ ]:
import shutil
shutil.make_archive(f"/content/{PROPERTY_ID}_results", "zip", f"out/{PROPERTY_ID}")
files.download(f"/content/{PROPERTY_ID}_results.zip")